# Document-Level Window Evaluation

This report aggregates continuous window MG-scores into one prediction per document. `Mixed+BBC` uses its own formed artifact sample. Common documents are selected within each evaluation regime AND scorer, across LR/RF and evaluation windows. Existing scorer cohorts have no global intersection, so cross-scorer metrics are descriptive and are not matched-document comparisons.

In [ ]:
from pathlib import Path
import os
import pandas as pd
from notebooks.utils.document_window_evaluation_utils import (
    DocumentWindowEvaluationConfig, evaluate_configurations, write_document_window_outputs,
    load_document_configurations,
)

In [ ]:

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
EVALUATION_RUN_ID = 'feature-vector-classification-2026-09-25-logreg-random-forest-max600-min100'
scorer_llms = ('gemma_3_270m_it', 'gemma_3_1b_it', 'littlelamb', 'smollm2_360m')
model_families = ('logistic_regression', 'random_forest')
evaluation_regimes = ('Kaggle+MAGE+Outfox', 'Kaggle+MAGE+Outfox+BBC')
training_window = 32
evaluation_windows = (32, 64)
sample_mode = "common"
common_group_columns = ('evaluation_regime', 'scorer_llm')
document_threshold = 0.5
config = DocumentWindowEvaluationConfig(
    run_id=EVALUATION_RUN_ID, scorer_llms=scorer_llms, model_families=model_families,
    training_regime='Kaggle+MAGE+Outfox', evaluation_regimes=evaluation_regimes,
    feature_set='simple_similarity', training_window=training_window,
    evaluation_windows=evaluation_windows, partition='test',
    document_threshold=document_threshold, sample_mode=sample_mode,
)

In [ ]:
prediction_root = DATA_ROOT / 'experiment-results' / 'classification' / 'evaluations' / config.run_id / 'predictions'
documents_by_configuration = load_document_configurations(
    prediction_root, scorer_llms=config.scorer_llms, model_families=config.model_families,
    evaluation_regimes=config.evaluation_regimes, evaluation_windows=config.evaluation_windows,
    training_window=config.training_window, training_regime=config.training_regime,
    feature_set=config.feature_set, partition=config.partition, document_threshold=config.document_threshold,
)
document_predictions, document_metrics, document_coverage = evaluate_configurations(
    documents_by_configuration, document_threshold=config.document_threshold, sample_mode=config.sample_mode,
    common_group_columns=common_group_columns,
)
output_dir = DATA_ROOT / 'experiment-results' / 'report' / 'document-window-evaluation'
if not document_predictions.empty:
    write_document_window_outputs(
        output_dir, document_predictions=document_predictions, document_metrics=document_metrics,
        document_coverage=document_coverage, manifest={
            'run_id': config.run_id, 'training_window': config.training_window,
            'evaluation_windows': config.evaluation_windows, 'sample_mode': config.sample_mode,
            'common_group_columns': common_group_columns,
            'document_threshold': config.document_threshold, 'aggregation': 'mean_window_score',
        }
    )

In [ ]:
display(document_metrics)
display(document_coverage)
display(document_predictions)